#Import Codes

In [ ]:
!pip install nltk pandas num2words tabulate datasets scikit-learn transformers accelerate tensorboard sentencepiece networkx matplotlib biopython spacy
#!pip install spacy==3.5.4

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.4 MB/s eta 0:00:00
  

In [ ]:
import gc
import os
import re
from functools import partial
import json


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tabulate import tabulate
from tqdm import tqdm


from Bio import Entrez
import nltk
import spacy
from num2words import num2words


import networkx as nx


from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertTokenizer,
    BertTokenizerFast,
    DataCollatorForSeq2Seq,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split


import accelerate
import torch
import torch.nn.functional as F

In [ ]:
login(token="your_HF_token_here")

SyntaxError: invalid syntax (<ipython-input-3-f67f8e96d4cc>, line 1)

# Load Datasets

##  Locally

In [ ]:
with open("/content/euadr_full.json", "r") as f:
    euadr_ds = json.load(f)

with open("/content/chemprot_train.json", "r") as f:
    chemprot_train = json.load(f)

with open("/content/chemprot_validation.json", "r") as f:
    chemprot_validation = json.load(f)

with open("/content/chemprot_test.json", "r") as f:
    chemprot_test = json.load(f)

chem_ds = {
    "train": chemprot_train,
    "validation": chemprot_validation,
    "test": chemprot_test
}



train_data = euadr_ds

## Online

In [ ]:
chem_ds = load_dataset("bigbio/chemprot", "chemprot_bigbio_kb")
euadr_ds = load_dataset("bigbio/euadr", "euadr_bigbio_kb")
train_data = euadr_ds["train"]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ConnectionError: Couldn't reach https://biocreative.bioinformatics.udel.edu/media/store/files/2017/ChemProt_Corpus.zip (error 403)

#NER (ChemProt)

##All Models (BioBERT/PubMedBERT/ClinicalBERT/SciBERT)

In [ ]:
# Define model variants
model_names = {
    "BioBERT": "dmis-lab/biobert-v1.1",
    "PubMedBERT": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    "ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "SciBERT": "allenai/scibert_scivocab_uncased",
}

# Map ChemProt types to simplified NER labels
entity_type_map = {
    "CHEMICAL": "Chemical",
    "GENE-Y": "Gene",
    "GENE-N": "Gene"
}
label_map = {"O": 0, "Chemical": 1, "Gene": 2}
id_to_label = {v: k for k, v in label_map.items()}

# Extract abstracts and their entities
abstracts = []
entity_annotations = []

chem_train = chem_ds["train"]

for entry in chem_train:
    abstract_passages = [p for p in entry["passages"] if p["type"] == "title and abstract"]
    if not abstract_passages:
        continue

    abstract_text = abstract_passages[0]["text"][0].strip()
    if not abstract_text:
        continue

    if entry["entities"]:
        abstracts.append(abstract_text)
        entity_annotations.append(entry["entities"])

# Encoding function
def encode_labels(text, entities, tokenizer):
    encodings = tokenizer(text, truncation=True, padding="max_length", return_offsets_mapping=True, max_length=512)
    word_ids = encodings.word_ids()
    labels = ["O"] * len(word_ids)

    for entity in entities:
        entity_start, entity_end = entity["offsets"][0]
        entity_type = entity_type_map.get(entity["type"], "O")
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                start, end = encodings["offset_mapping"][idx]
                if start >= entity_start and end <= entity_end:
                    labels[word_id] = entity_type

    label_ids = []
    for word_id in word_ids:
        if word_id is None:
            label_ids.append(-100)
        elif word_id < len(labels):
            label_ids.append(label_map.get(labels[word_id], 0))
        else:
            label_ids.append(0)

    encodings.pop("offset_mapping")
    encodings["labels"] = label_ids
    return encodings

In [ ]:
# Evaluation function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions, true_labels = [], []
    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="weighted", zero_division=1
    )
    accuracy = accuracy_score(true_labels, true_predictions)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Loop through each model
for model_name, model_path in model_names.items():
    print(f"\nTraining {model_name}...")

    tokenizer = BertTokenizerFast.from_pretrained(model_path)

    # Tokenize the dataset for this model
    tokenized_data = [
        encode_labels(text, entities, tokenizer)
        for text, entities in zip(abstracts, entity_annotations)
    ]

    train_data, val_data = train_test_split(tokenized_data, test_size=0.2, random_state=42)

    train_dataset = Dataset.from_dict({
        "input_ids": [x['input_ids'] for x in train_data],
        "attention_mask": [x['attention_mask'] for x in train_data],
        "labels": [x['labels'] for x in train_data]
    })

    val_dataset = Dataset.from_dict({
        "input_ids": [x['input_ids'] for x in val_data],
        "attention_mask": [x['attention_mask'] for x in val_data],
        "labels": [x['labels'] for x in val_data]
    })

    data_collator = DataCollatorForTokenClassification(tokenizer)

    model = BertForTokenClassification.from_pretrained(model_path, num_labels=len(label_map))

    training_args = TrainingArguments(
        output_dir=f'./results/{model_name}_chemprot_ner',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=f'./logs/{model_name}_chemprot_ner',
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        report_to="tensorboard",
        run_name=f"{model_name}_NER_ChemProt",
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="eval_loss",
        greater_is_better=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    print(f"Finished training {model_name} on ChemProt!\n")

print("✅ All models trained on ChemProt NER!")


Training BioBERT...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-07c7abb1c456>:68: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# NER (EU-ADR)

## Single Model (BioBERT)

In [ ]:
# Load BioBERT tokenizer
tokenizer = BertTokenizer.from_pretrained("dmis-lab/biobert-v1.1", do_lower_case=True)

# Label mapping
label_map = {
    "Diseases & Disorders": 1,
    "Chemicals & Drugs": 2,
    "Gene/ENA": 3,
    "Protein": 4,
    "Variation": 5,
    "O": 0
}

# Text cleaning utilities
def convert_numbers_to_words(text):
    return re.sub(r'\b\d+\b', lambda x: num2words(int(x.group())), text)

def clean_text(text):
    text = convert_numbers_to_words(text)
    text = re.sub(r"[^a-zA-Z0-9' percent ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

# Step 1: Extract and clean abstracts + entity annotations
abstracts, entity_annotations = [], []

for entry in euadr_ds:
    abstract_text = ""
    for passage in entry["passages"]:
        if passage["type"] == "abstract":
            abstract_text = passage["text"][0]
            break
    if abstract_text:
        cleaned = clean_text(abstract_text)
        abstracts.append(cleaned)
        entity_annotations.append(entry["entities"])

# Step 2: Generate token-level labels
def generate_labels(text, entities):
    tokens = tokenizer.tokenize(text)
    labels = ['O'] * len(tokens)

    for entity in entities:
        entity_text = entity["text"][0].lower()
        entity_type = entity["type"]
        entity_tokens = tokenizer.tokenize(entity_text)

        for i in range(len(tokens) - len(entity_tokens) + 1):
            if tokens[i:i + len(entity_tokens)] == entity_tokens:
                for j in range(len(entity_tokens)):
                    labels[i + j] = entity_type
                break

    return [label_map.get(label, 0) for label in labels]

# Step 3: Tokenize and label all data
tokenized_data, all_labels = [], []

for abstract, entities in zip(abstracts, entity_annotations):
    encoded = tokenizer(abstract, padding=True, truncation=True, return_tensors="pt")
    tokenized_data.append(encoded)
    all_labels.append(generate_labels(abstract, entities))

# Step 4: Convert to Hugging Face Dataset
train_data = {
    'input_ids': [data['input_ids'].squeeze().tolist() for data in tokenized_data],
    'attention_mask': [data['attention_mask'].squeeze().tolist() for data in tokenized_data],
    'labels': all_labels
}

train_dataset = Dataset.from_dict(train_data)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("dmis-lab/biobert-v1.1", do_lower_case=True)

label_map = {'O': 0, 'Gene/RNA': 1, 'Protein': 2, 'Variation': 3, 'Disease': 4, 'Drug': 5}
id_to_label = {v: k for k, v in label_map.items()}

assert len(abstracts) == len(entity_annotations)

def encode_labels(text, entities, tokenizer):
    """
    Tokenizes text and aligns entity labels with tokenized words.
    """

    encodings = tokenizer(text, truncation=True, padding="max_length", return_offsets_mapping=True, max_length=512)

    word_ids = encodings.word_ids()
    labels = ["O"] * len(text.split())


    for entity in entities:
        entity_text = entity["text"][0].lower()
        entity_type = entity["type"]
        entity_start, entity_end = entity["offsets"][0]


        for idx, word_id in enumerate(word_ids):
            if word_id is not None and word_id < len(labels):
                start, end = encodings["offset_mapping"][idx]
                if start >= entity_start and end <= entity_end:
                    labels[word_id] = entity_type


    label_ids = []
    for word_id in word_ids:
        if word_id is None:
            label_ids.append(-100)
        elif word_id < len(labels):
            label_ids.append(label_map.get(labels[word_id], 0))
        else:
            label_ids.append(0)

    encodings.pop("offset_mapping")
    encodings["labels"] = label_ids
    return encodings




tokenized_data = [encode_labels(text, entities, tokenizer) for text, entities in zip(abstracts, entity_annotations)]

In [ ]:
train_data, val_data = train_test_split(tokenized_data, test_size=0.2, random_state=42)


train_dataset = Dataset.from_dict({
    "input_ids": [x['input_ids'] for x in train_data],
    "attention_mask": [x['attention_mask'] for x in train_data],
    "labels": [x['labels'] for x in train_data]
})

val_dataset = Dataset.from_dict({
    "input_ids": [x['input_ids'] for x in val_data],
    "attention_mask": [x['attention_mask'] for x in val_data],
    "labels": [x['labels'] for x in val_data]
})


model = BertForTokenClassification.from_pretrained("dmis-lab/biobert-v1.1", num_labels=len(label_map))

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="weighted", zero_division=1
    )
    accuracy = accuracy_score(true_labels, true_predictions)

    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}


training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="tensorboard",
    run_name="BIOBert_NER_2",
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.910700,1.794779,0.177944,1.000000,0.177944,0.302127
2,1.519800,1.361347,0.847750,1.000000,0.847750,0.917602
3,1.108200,0.528188,0.999947,1.000000,0.999947,0.999973
4,0.072700,0.014061,1.000000,1.000000,1.000000,1.000000
5,0.009800,0.002254,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=75, training_loss=0.9218899899969498, metrics={'train_runtime': 36.1804, 'train_samples_per_second': 33.167, 'train_steps_per_second': 2.073, 'total_flos': 313567447449600.0, 'train_loss': 0.9218899899969498, 'epoch': 5.0})

In [ ]:
model.save_pretrained("./saved_models/ner_model")
tokenizer.save_pretrained("./saved_models/ner_tokenizer")

('./saved_models/ner_tokenizer/tokenizer_config.json',
 './saved_models/ner_tokenizer/special_tokens_map.json',
 './saved_models/ner_tokenizer/vocab.txt',
 './saved_models/ner_tokenizer/added_tokens.json',
 './saved_models/ner_tokenizer/tokenizer.json')

## All Four Models (BioBERT/PubMedBERT/ClinicalBERT/SciBERT)

In [ ]:
model_names = {
    "BioBERT": "dmis-lab/biobert-v1.1",
    "PubMedBERT": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
    "ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "SciBERT": "allenai/scibert_scivocab_uncased",
}


In [ ]:
# Define label mappings
label_map = {'O': 0, 'Gene/RNA': 1, 'Protein': 2, 'Variation': 3, 'Disease': 4, 'Drug': 5}
id_to_label = {v: k for k, v in label_map.items()}

# Function to tokenize and align labels
def encode_labels(text, entities, tokenizer):
    encodings = tokenizer(text, truncation=True, padding="max_length", return_offsets_mapping=True, max_length=512)
    word_ids = encodings.word_ids()
    labels = ["O"] * len(word_ids)  # Initialize labels

    # Assign entity labels
    for entity in entities:
        entity_type = entity["type"]
        entity_start, entity_end = entity["offsets"][0]

        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                start, end = encodings["offset_mapping"][idx]
                if start >= entity_start and end <= entity_end:
                    labels[word_id] = entity_type

    # Convert labels to IDs
    label_ids = []
    for word_id in word_ids:
        if word_id is None:
            label_ids.append(-100)  # Ignore special tokens
        elif word_id < len(labels):
            label_ids.append(label_map.get(labels[word_id], 0))
        else:
            label_ids.append(0)

    encodings.pop("offset_mapping")
    encodings["labels"] = label_ids
    return encodings


# Tokenize dataset
tokenized_data = [encode_labels(text, entities, BertTokenizerFast.from_pretrained("dmis-lab/biobert-v1.1"))
                  for text, entities in zip(abstracts, entity_annotations)]

# Split into train and validation sets
train_data, val_data = train_test_split(tokenized_data, test_size=0.2, random_state=42)

train_dataset = Dataset.from_dict({
    "input_ids": [x['input_ids'] for x in train_data],
    "attention_mask": [x['attention_mask'] for x in train_data],
    "labels": [x['labels'] for x in train_data]
})

val_dataset = Dataset.from_dict({
    "input_ids": [x['input_ids'] for x in val_data],
    "attention_mask": [x['attention_mask'] for x in val_data],
    "labels": [x['labels'] for x in val_data]
})

In [ ]:
# Define evaluation function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions, true_labels = [], []
    for pred, label in zip(predictions, labels):
        for p, l in zip(pred, label):
            if l != -100:
                true_predictions.append(p)
                true_labels.append(l)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_predictions, average="weighted", zero_division=1
    )
    accuracy = accuracy_score(true_labels, true_predictions)
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# Set up data collator for padding
data_collator = DataCollatorForTokenClassification(BertTokenizerFast.from_pretrained("dmis-lab/biobert-v1.1"))

# Train models in a loop
for model_name, model_path in model_names.items():
    print(f"Training {model_name}...")

    tokenizer = BertTokenizerFast.from_pretrained(model_path)
    model = BertForTokenClassification.from_pretrained(model_path, num_labels=len(label_map))

    training_args = TrainingArguments(
        output_dir=f'./results/{model_name}',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir=f'./logs/{model_name}',
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        report_to="tensorboard",
        run_name=f"{model_name}_NER_2",  # Unique name for TensorBoard
        load_best_model_at_end=True,
        save_total_limit=2,
        metric_for_best_model="eval_loss",
        greater_is_better=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    print(f"Finished training {model_name}!\n")

print("All models trained successfully!")

Training BioBERT...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.702100,1.561214,0.380466,1.000000,0.380466,0.551214


KeyboardInterrupt: 

# Relation Extraction

## ChemProt

In [ ]:
# === Relation mapping ===
target_relations = {
    "Downregulator": 0,
    "Agonist": 1,
    "Antagonist": 2,
    "Part_of": 3,
    "Regulator": 4,
    "Substrate": 5,
    "Not": 6,
    "Upregulator": 7,
    "Modulator": 8,
    "Cofactor": 9
}

# === Format helpers ===
def format_sentence(text, entity1, entity2, entity1_offset, entity2_offset):
    if entity1_offset[0] > entity2_offset[0]:
        entity1, entity2 = entity2, entity1
        entity1_offset, entity2_offset = entity2_offset, entity1_offset

    return (
        text[:entity1_offset[0]] +
        f"[E1] {entity1} [/E1]" +
        text[entity1_offset[1]:entity2_offset[0]] +
        f"[E2] {entity2} [/E2]" +
        text[entity2_offset[1]:]
    )

def get_expanded_sentence(text, entity1_offset, entity2_offset, window=2):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    start_idx = end_idx = None

    for i, sentence in enumerate(sentences):
        sentence_start = text.index(sentence)
        sentence_end = sentence_start + len(sentence)
        if entity1_offset[0] in range(sentence_start, sentence_end) or entity2_offset[0] in range(sentence_start, sentence_end):
            if start_idx is None:
                start_idx = max(0, i - window)
            end_idx = min(len(sentences), i + window + 1)

    if start_idx is not None and end_idx is not None:
        return " ".join(sentences[start_idx:end_idx])
    return None

# === Extract relations from ChemProt dataset ===
def extract_chem_data(dataset):
    data = []
    for row in dataset:
        text = row["passages"][0]["text"][0]
        entity_dict = {
            e["id"]: (e["text"][0], e["offsets"][0])
            for e in row["entities"]
        }
        for rel in row["relations"]:
            e1_id, e2_id, rel_type = rel["arg1_id"], rel["arg2_id"], rel["type"]
            if e1_id not in entity_dict or e2_id not in entity_dict:
                continue
            e1_text, e1_offset = entity_dict[e1_id]
            e2_text, e2_offset = entity_dict[e2_id]
            sentence = get_expanded_sentence(text, e1_offset, e2_offset)
            if sentence is None:
                continue
            formatted = format_sentence(sentence, e1_text, e2_text, e1_offset, e2_offset)
            data.append({"sentence": formatted, "relation": rel_type})
    return pd.DataFrame(data)

In [ ]:
relation_counts = {
    "Downregulator": 5030,
    "Agonist": 487,
    "Antagonist": 727,
    "Part_of": 676,
    "Regulator": 4175,
    "Substrate": 1828,
    "Not": 683,
    "Upregulator": 1996,
    "Modulator": 73,
    "Cofactor": 61,
}

total = sum(relation_counts.values())
weights = {label: total / count for label, count in relation_counts.items()}

# Map weights to the correct order based on target_relations
class_weights = [weights[label] for label in sorted(target_relations, key=target_relations.get)]
class_weights_tensor = torch.tensor(class_weights).to("cuda")



class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
chem_train, chem_test, chem_val = chem_ds["train"], chem_ds["test"], chem_ds["validation"]
chem_train_df = extract_chem_data(chem_train)
chem_val_df = extract_chem_data(chem_val)

chem_train_df = chem_train_df[chem_train_df["relation"].isin(target_relations.keys())]
chem_val_df = chem_val_df[chem_val_df["relation"].isin(target_relations.keys())]

chem_train_df["label"] = chem_train_df["relation"].map(target_relations)
chem_val_df["label"] = chem_val_df["relation"].map(target_relations)

train_dataset = Dataset.from_pandas(chem_train_df)
val_dataset = Dataset.from_pandas(chem_val_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})


<ipython-input-7-4fe0bad0ae33>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chem_train_df["label"] = chem_train_df["relation"].map(target_relations)
<ipython-input-7-4fe0bad0ae33>:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chem_val_df["label"] = chem_val_df["relation"].map(target_relations)


In [ ]:
# === Metric function ===
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}


In [ ]:

models = {
    "LLaMA_2": "meta-llama/Llama-2-7b-hf",
    "Falcon_7B": "tiiuae/falcon-7b",
    "Falcon_1B": "tiiuae/falcon-rw-1b",
    "TinyLLaMA": "TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0",
    "LLaMA_2": "TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0",
    "SciBERT" : "allenai/scibert_scivocab_uncased",
    "BioGPT" : "microsoft/biogpt",
}

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

for model_name, model_path in models.items():
    gc.collect()
    torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or "[PAD]"

    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=len(target_relations)
    ).to("cuda")
    model.config.use_cache = False

    def tokenize_function(examples):
        encoding = tokenizer(
            examples["sentence"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_token_type_ids=False
        )
        encoding["labels"] = [target_relations.get(rel, 6) for rel in examples["relation"]]
        return encoding

    tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir=f"./results/{model_name}_ChemProt_RE",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=3,
        dataloader_num_workers=1,
        logging_steps=50,
        save_total_limit=1,
        num_train_epochs=1,
        weight_decay=0.01,
        run_name=f"{model_name}_ChemProt_RE_V2",
        logging_dir=f"./logs/{model_name}",
        fp16=True,
        bf16=False,
        optim="adafactor",
        load_best_model_at_end=True,
        report_to="tensorboard",
        remove_unused_columns=True
    )

    trainer = WeightedLossTrainer(
      model=model,
      args=training_args,
      train_dataset=tokenized_dataset["train"],
      eval_dataset=tokenized_dataset["validation"],
      compute_metrics=compute_metrics,
      callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
      class_weights=class_weights_tensor
    )

    trainer.train()
    trainer.save_model(f"./saved_models/{model_name}_ChemProt_RE")
    tokenizer.save_pretrained(f"./saved_models/{model_name}_ChemProt_RE")


    chem_test_df = extract_chem_data(chem_test)
    chem_test_df = chem_test_df[chem_test_df["relation"].isin(target_relations.keys())]
    chem_test_df["label"] = chem_test_df["relation"].map(target_relations)
    test_dataset = Dataset.from_pandas(chem_test_df)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)

    test_results = trainer.evaluate(eval_dataset=tokenized_test)
    print(f"\n Test Results for {model_name}:", test_results)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Completed all model training")

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/6436 [00:00<?, ? examples/s]

Map:   0%|          | 0/3554 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## EU-ADR

In [ ]:
def extract_entities(dataset):
    entity_dict = {}

    for row in dataset:
        for entity in row.get("entities", []):

            entity_id = entity.get("id", None)
            entity_text = entity.get("text", [""])[0]
            entity_type = entity.get("type", "Unknown")

            if entity_id:
                if entity_id not in entity_dict:
                    entity_dict[entity_id] = []
                entity_dict[entity_id].append({"text": entity_text, "type": entity_type})

    return entity_dict

entity_dict = extract_entities(euadr_ds)

def extract_relations(dataset, entity_dict):
    """
    Extracts relations between entities in all rows.
    Matches entity pairs to their relationship type.
    """
    relations = []

    for row in dataset:
        for rel in row.get("relations", []):
            arg1_id = rel.get("arg1_id")
            arg2_id = rel.get("arg2_id")
            rel_type = rel.get("type", "Unknown")

            if arg1_id in entity_dict and arg2_id in entity_dict:
                for entity1 in entity_dict[arg1_id]:
                    for entity2 in entity_dict[arg2_id]:
                        relations.append({
                            "entity1_text": entity1["text"],
                            "entity1_type": entity1["type"],
                            "entity2_text": entity2["text"],
                            "entity2_type": entity2["type"],
                            "relation": rel_type
                        })

    return relations

relations = extract_relations(euadr_ds, entity_dict)


def format_sentence(text, entity1, entity2, entity1_offset, entity2_offset):
    if entity1_offset[0] > entity2_offset[0]:
        entity1, entity2 = entity2, entity1
        entity1_offset, entity2_offset = entity2_offset, entity1_offset


    modified_text = (
        text[:entity1_offset[0]] +
        f"[E1] {entity1} [/E1]" +
        text[entity1_offset[1]:entity2_offset[0]] +
        f"[E2] {entity2} [/E2]" +
        text[entity2_offset[1]:]
    )

    return modified_text

def get_expanded_sentence(text, entity1_offset, entity2_offset, window=2):

    sentences = re.split(r'(?<=[.!?])\s+', text)
    start_idx = None
    end_idx = None


    for i, sentence in enumerate(sentences):
        sentence_start = text.index(sentence)
        sentence_end = sentence_start + len(sentence)


        if entity1_offset[0] in range(sentence_start, sentence_end) or entity2_offset[0] in range(sentence_start, sentence_end):
            if start_idx is None:
                start_idx = max(0, i - window)
            end_idx = min(len(sentences), i + window + 1)


    if start_idx is not None and end_idx is not None:
        return " ".join(sentences[start_idx:end_idx])

    return None


data = []

for row in euadr_ds:
    text = row["passages"][1]["text"][0]
    entity_dict = {e["id"]: (e["text"][0], e["offsets"][0]) for e in row["entities"]}

    for rel in row["relations"]:
        e1, e2 = rel["arg1_id"], rel["arg2_id"]
        if e1 not in entity_dict or e2 not in entity_dict:
            continue

        e1_text, e1_offset = entity_dict[e1]
        e2_text, e2_offset = entity_dict[e2]

        sentence = get_expanded_sentence(text, e1_offset, e2_offset)
        if not sentence:
            continue

        data.append({
            "sentence": format_sentence(sentence, e1_text, e2_text, e1_offset, e2_offset),
            "relation": rel["type"]
        })



df = pd.DataFrame(data)

In [ ]:
models = {
    "LLaMA_2": "meta-llama/Llama-2-7b-hf",
    "Falcon_7B": "tiiuae/falcon-7b",
    "Falcon_1B": "tiiuae/falcon-rw-1b",
    "TinyLLaMA": "TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0",
    "LLaMA_2": "TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0",
    "SciBERT" : "allenai/scibert_scivocab_uncased",
    "BioGPT" : "microsoft/biogpt",
}


relation_map = {
    "PA": 0,
    "NA": 1,
    "SA": 2
}


def tokenize_function(examples, tokenizer):
    """
    Tokenizes relation extraction dataset for LLaMA 2 or Falcon.
    Uses <s> special tokens for formatted input.
    """
    modified_sentences = [
        f"<s> Entity1: {e1} | Entity2: {e2} | Relation: {rel} </s>"
        for e1, e2, rel in zip(examples["sentence"], examples["relation"], examples["relation"])
    ]

    encoding = tokenizer(
        modified_sentences,
        truncation=True,
        padding="max_length",
        max_length=512
    )
    encoding["labels"] = [relation_map.get(rel, -1) for rel in examples["relation"]]
    return encoding

In [ ]:
dataset = Dataset.from_pandas(df)


train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})



def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=1
    )
    accuracy = accuracy_score(labels, predictions)

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


def tokenize_function(examples, tokenizer):
  encoding = tokenizer(
    examples["sentence"],
    truncation=True,
    padding="max_length",
    max_length=512
  )
  encoding["labels"] = [relation_map.get(rel, -1) for rel in examples["relation"]]
  return encoding

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


free_mem = torch.cuda.mem_get_info()[0] / 1024**2


for model_name, model_path in models.items():
    print(f"\nTraining {model_name}")


    gc.collect()
    torch.cuda.empty_cache()
    print(torch.cuda.memory_summary(device=0, abbreviated=True))

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or "[PAD]"


    model = AutoModelForSequenceClassification.from_pretrained(
        model_path,
        num_labels=len(relation_map),
    )

    model.config.use_cache = False
    model = model.to("cuda")


    default_label = relation_map.get("NO_RELATION", 0)

    def tokenize_function(examples):
        encoding = tokenizer(
            examples["sentence"],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_token_type_ids=False
        )
        encoding["labels"] = [
            relation_map.get(rel, default_label) for rel in examples["relation"]
        ]
        return encoding

    tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

    training_args = TrainingArguments(
        output_dir=f"./results/{model_name}_NER",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=3,
        dataloader_num_workers=1,
        logging_steps=50,
        save_total_limit=1,
        num_train_epochs=1,
        weight_decay=0.01,
        run_name=f"{model_name}_RE_EUADR",
        logging_dir=f"./logs/{model_name}",
        fp16=True,
        bf16=False,
        optim="adafactor",
        load_best_model_at_end=True,
        report_to="tensorboard",
        remove_unused_columns=True
    )


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    trainer.save_model(f"./saved_models/{model_name}_EUADR")
    tokenizer.save_pretrained(f"./saved_models/{model_name}_EUADR")
    print(f"Finished training and saved {model_name}!")


    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("All models trained and saved")


Free GPU memory before starting: 27458.88 MiB

🔥 Training TinyLLaMA...
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   7913 MiB |  11787 MiB |   2710 GiB |   2703 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   7913 MiB |  11787 MiB |   2710 GiB |   2703 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |   7912 MiB |  11768 MiB |   2709 GiB |   2701

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at TinyLLaMA/TinyLLaMA-1.1B-Chat-v1.0 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/2184 [00:00<?, ? examples/s]

Map:   0%|          | 0/547 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.742100,0.939458,0.851920,0.800530,0.851920,0.791848


Finished training and saved TinyLLaMA!
All models trained and saved


# Test (ChemProt)

## Per Class Metrics

In [ ]:
# === Load model and tokenizer ===
model_dir = "./saved_models/TinyLLaMA_ChemProt_RE"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

model = AutoModelForSequenceClassification.from_pretrained(model_dir).to("cuda")
model.eval()

# === Relation label mappings ===
target_relations = {
    "Downregulator": 0,
    "Agonist": 1,
    "Antagonist": 2,
    "Part_of": 3,
    "Regulator": 4,
    "Substrate": 5,
    "Not": 6,
    "Upregulator": 7,
    "Modulator": 8,
    "Cofactor": 9
}
id2rel = {v: k for k, v in target_relations.items()}

# === Sentence formatting helpers ===
def format_sentence(text, entity1, entity2, offset1, offset2):
    if offset1[0] > offset2[0]:
        entity1, entity2 = entity2, entity1
        offset1, offset2 = offset2, offset1
    return (
        text[:offset1[0]] +
        f"[E1] {entity1} [/E1]" +
        text[offset1[1]:offset2[0]] +
        f"[E2] {entity2} [/E2]" +
        text[offset2[1]:]
    )

# === Format and collect gold label examples from chem_train ===
true_labels = []
pred_labels = []
batch_sentences = []


for row in tqdm(chem_train):
    text = row["passages"][0]["text"][0]
    entity_dict = {
        e["id"]: (e["text"][0], e["offsets"][0]) for e in row["entities"]
    }

    for rel in row["relations"]:
        e1_id, e2_id, rel_type = rel["arg1_id"], rel["arg2_id"], rel["type"]
        if rel_type not in target_relations:
            continue
        if e1_id not in entity_dict or e2_id not in entity_dict:
            continue

        (text1, offset1) = entity_dict[e1_id]
        (text2, offset2) = entity_dict[e2_id]
        formatted = format_sentence(text, text1, text2, offset1, offset2)
        batch_sentences.append(formatted)
        true_labels.append(target_relations[rel_type])

# === Run inference in batches ===
batch_size = 128
for i in tqdm(range(0, len(batch_sentences), batch_size)):
    batch = batch_sentences[i:i+batch_size]
    inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=128).to("cuda")

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, axis=1).tolist()
        pred_labels.extend(preds)

print("\nPer-class F1 Score Report (chem_train):\n")
print(classification_report(true_labels, pred_labels, target_names=[id2rel[i] for i in range(len(id2rel))]))

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': './saved_models/TinyLLaMA_ChemProt_RE'. Use `repo_type` argument if needed.

## New Relations

In [ ]:

# Load ChemProt dataset from earlier split
chemprot_ds = chem_ds
chemprot_split = chemprot_ds["train"]  # You can also switch to "validation" or "test"

# Load model and tokenizer
model_path = "./saved_models/TinyLLaMA_ChemProt_RE"
model = AutoModelForCausalLM.from_pretrained(model_path).cuda().eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

# Target relations and lowercase matching set
target_relations = {
    "Downregulator": 0, "Agonist": 1, "Antagonist": 2, "Part_of": 3,
    "Regulator": 4, "Substrate": 5, "Not": 6, "Upregulator": 7,
    "Modulator": 8, "Cofactor": 9
}
id_to_label = {v: k for k, v in target_relations.items()}

# For cleaning and matching predictions
relation_map = {
    key.lower().replace("_", "").replace(" ", ""): key
    for key in target_relations.keys()
}

known_types_str = "upregulate, downregulate, agonist, part of, substrate, modulate, cofactor, not"

results = []

for row in tqdm(chemprot_split, desc="🔍 Finding new relations (ChemProt)", unit="abstract"):

    passages = [p["text"][0] for p in row["passages"] if p["type"] == "abstract"]
    if not passages:
        continue
    sentence = passages[0]

    entities = {
        e["id"]: {"text": e["text"][0], "offset": e["offsets"][0]}
        for e in row["entities"]
    }
    entity_ids = list(entities.keys())

    for i in range(len(entity_ids)):
        for j in range(i + 1, len(entity_ids)):
            e1_id, e2_id = entity_ids[i], entity_ids[j]
            e1, e2 = entities[e1_id], entities[e2_id]


            # Skip if this pair is already annotated
            is_existing_relation = False
            for rel in row.get("relations", []):
                if (
                    (rel["arg1_id"] == e1_id and rel["arg2_id"] == e2_id) or
                    (rel["arg1_id"] == e2_id and rel["arg2_id"] == e1_id)
                ):
                    is_existing_relation = True
                    break
            if is_existing_relation:
                continue  # skip known relations

            # Build prompt
            prompt = (
                f"Sentence: \"{sentence}.\"\n"
                f"Entities: ({e1['text']}, {e2['text']})\n\n"
                f"What is the relation between these two entities? "
                f"Known types: {known_types_str}.\n"
            )

            # Tokenize and run model
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                               padding="max_length", max_length=512)
            inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=10,
                    do_sample=False,
                    return_dict_in_generate=True,
                    output_scores=False
                )
                decoded = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
                prediction = decoded.replace(prompt, "").strip()

            # Normalize prediction
            cleaned = prediction.lower().replace("-", "").replace(" ", "").replace("_", "")
            predicted_relation = relation_map.get(cleaned, "Not")

            if predicted_relation == "Not":
                continue  # skip non-relations

            results.append({
                "sentence": sentence,
                "entity1": e1["text"],
                "entity2": e2["text"],
                "prompt": prompt,
                "raw_prediction": prediction,
                "predicted_relation": predicted_relation
            })

# Save new relation predictions
df_results = pd.DataFrame(results)
df_results.to_csv("chemprot_new_relations.csv", index=False)


Some weights of LlamaForCausalLM were not initialized from the model checkpoint at ./saved_models/TinyLLaMA_ChemProt_RE and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
🔍 Finding new relations (ChemProt): 100%|██████████| 1020/1020 [00:00<00:00, 42370.48abstract/s]

Abstract 1: 5 entities, 10 total pairs so far
Abstract 2: 29 entities, 416 total pairs so far
Abstract 3: 14 entities, 507 total pairs so far
Abstract 4: 33 entities, 1035 total pairs so far
Abstract 5: 19 entities, 1206 total pairs so far
Abstract 6: 15 entities, 1311 total pairs so far
Abstract 7: 14 entities, 1402 total pairs so far
Abstract 8: 18 entities, 1555 total pairs so far
Abstract 9: 16 entities, 1675 total pairs so far
Abstract 10: 41 entities, 2495 total pairs so far
Abstract 11: 71 entities, 4980 total pairs so far
Abstract 12: 16 entities, 5100 total pairs so far
Abstract 13: 29 entities, 5506 total pairs so far
Abstract 14: 25 entities, 5806 total pairs so far
Abstract 15: 15 entities, 5911 total pairs so far
Abstract 16: 15 entities, 6016 total pairs so far
Abstract 17: 57 entities, 7612 total pairs so far
Abstract 18: 12 entities, 7678 total pairs so far
Abstract 19: 9 entities, 7714 total pairs so far
Abstract 20: 31 entities, 8179 total pairs so far
Abstract 21: 16

## Graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        e1, e2, rel = row["entity1"], row["entity2"], row["predicted_relation"]
        if not G.has_edge(e1, e2):
            G.add_edge(e1, e2, relation=rel)
    return G

def get_k_hop_pairs(G, k):
    pairs = set()
    for source in G.nodes:
        for target in G.nodes:
            if source != target and nx.has_path(G, source, target):
                try:
                    if len(nx.shortest_path(G, source, target)) - 1 == k:
                        pairs.add((source, target))
                except:
                    continue
    return pairs


baseline_size = 25
step_size = 25
max_size = 400

hop_growth = {2: [], 3: [], 4: []}
sample_sizes = []

df_baseline = df_results.iloc[:baseline_size]
G_base = build_graph(df_baseline)
base_hops = {k: get_k_hop_pairs(G_base, k) for k in [2, 3, 4]}

for end in tqdm(range(baseline_size + step_size, max_size + 1, step_size), desc="Graph Progress"):
    df_chunk = df_results.iloc[:end]
    G = build_graph(df_chunk)

    for k in [2, 3, 4]:
        new_hops = get_k_hop_pairs(G, k)
        baseline_set = base_hops[k]
        percent_growth = (len(new_hops - baseline_set) / max(1, len(baseline_set))) * 100
        hop_growth[k].append(percent_growth)

    sample_sizes.append(end)

# Plotting
for k in [2, 3, 4]:
    plt.plot(sample_sizes, hop_growth[k], label=f"{k}-Hop")

plt.title("Multi-Hop Growth from ChemProt")
plt.xlabel("# of samples")
plt.ylabel("% of new history relations")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Test (EU-ADR)

In [ ]:
model_path = "./saved_models/TinyLLaMA_NER"
model = AutoModelForSequenceClassification.from_pretrained(model_path).cuda().eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)


id_to_label = {0: "PA", 1: "NA", 2: "SA"}

results = []

for row in tqdm(euadr_ds, desc="🔍 Predicting relations (EU-ADR)", unit="abstract"):
    passages = [p["text"][0] for p in row["passages"] if p["type"] == "abstract"]
    if not passages:
        continue
    text = passages[0]

    entities = {e["id"]: {"text": e["text"][0], "offset": e["offsets"][0]} for e in row["entities"]}
    entity_ids = list(entities.keys())

    for i in range(len(entity_ids)):
        for j in range(i + 1, len(entity_ids)):
            e1 = entities[entity_ids[i]]
            e2 = entities[entity_ids[j]]


            e1_start, e1_end = e1["offset"]
            e2_start, e2_end = e2["offset"]
            if e1_start > e2_start:
                e1, e2 = e2, e1
                e1_start, e1_end, e2_start, e2_end = e2_start, e2_end, e1_start, e1_end

            tagged_text = (
                text[:e1_start] +
                f"[E1] {e1['text']} [/E1]" +
                text[e1_end:e2_start] +
                f"[E2] {e2['text']} [/E2]" +
                text[e2_end:]
            )

            inputs = tokenizer(tagged_text, return_tensors="pt", truncation=True,
                               padding="max_length", max_length=256)
            inputs = {k: v.cuda() for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                probs = F.softmax(outputs.logits, dim=1)
                pred_id = torch.argmax(probs, dim=1).item()
                pred_label = id_to_label[pred_id]

            results.append({
                "sentence": tagged_text,
                "entity1": e1["text"],
                "entity2": e2["text"],
                "predicted_relation": pred_label
            })

df_results = pd.DataFrame(results)

🔍 Predicting relations (EU-ADR):   1%|          | 2/300 [00:36<1:30:35, 18.24s/abstract]


KeyboardInterrupt: 

## Graph

In [ ]:

def build_graph(df):
    G = nx.Graph()
    for _, row in df.iterrows():
        e1, e2, rel = row["entity1"], row["entity2"], row["predicted_relation"]
        G.add_edge(e1, e2, relation=rel)
    return G

def get_k_hop_pairs(G, k):
    pairs = set()
    for source in G.nodes:
        for target in G.nodes:
            if source != target and nx.has_path(G, source, target):
                try:
                    if len(nx.shortest_path(G, source, target)) - 1 == k:
                        pairs.add((source, target))
                except:
                    continue
    return pairs


baseline_size = 25
step_size = 25
max_size = 400

hop_growth = {2: [], 3: [], 4: []}
sample_sizes = []


df_baseline = df_results.iloc[:baseline_size]
G_base = build_graph(df_baseline)
base_hops = {k: get_k_hop_pairs(G_base, k) for k in [2, 3, 4]}


for end in tqdm(range(baseline_size + step_size, max_size + 1, step_size), desc="Graph Progress"):
    df_chunk = df_results.iloc[:end]
    G = build_graph(df_chunk)

    for k in [2, 3, 4]:
        new_hops = get_k_hop_pairs(G, k)
        baseline_set = base_hops[k]
        percent_growth = (len(new_hops - baseline_set) / max(1, len(baseline_set))) * 100
        hop_growth[k].append(percent_growth)

    sample_sizes.append(end)


for k in [2, 3, 4]:
    plt.plot(sample_sizes, hop_growth[k], label=f"{k}-Hop")

plt.title("Multi-Hop Growth from EU-ADR")
plt.xlabel("# of samples")
plt.ylabel("% of new history relations")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# PubMed Cross Verification

## Skeleton Code

In [ ]:
Entrez.email = "brian.c.tran@sjsu.edu"

def search_pubmed(entity1, entity2, max_results=10):
    query = f"{entity1} AND {entity2}"
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        id_list = record["IdList"]

        if not id_list:
            print(f"No articles found for: {entity1} AND {entity2}")
            return

        print(f"Found {len(id_list)} articles for: {entity1} AND {entity2}")
        print("Fetching summaries...")

        handle = Entrez.esummary(db="pubmed", id=",".join(id_list))
        summaries = Entrez.read(handle)
        handle.close()

        for i, doc in enumerate(summaries, start=1):
            print(f"\n🔹 Article {i}:")
            print(f"Title: {doc['Title']}")
            print(f"PubDate: {doc['PubDate']}")
            print(f"Source: {doc['Source']}")
            print(f"PMID: {doc['Id']}")

    except HTTPError as e:
        print(f"Error fetching results: {e}")

# Example usage
if __name__ == "__main__":
    e1 = input("entity1").strip()
    e2 = input("entity2").strip()
    search_pubmed(e1, e2)

entity1dsd
entity2dsds
Found 10 articles for: dsd AND dsds
Fetching summaries...

🔹 Article 1:
Title: An evaluation of cases of disorders of sex development related to SRD5A2.
PubDate: 2025 May 17
Source: Endocrine
PMID: 40381132

🔹 Article 2:
Title: The clinical diversity and molecular etiology in 46, XY disorders of sex development patients without uterus.
PubDate: 2025 Apr 17
Source: Orphanet J Rare Dis
PMID: 40247401

🔹 Article 3:
Title: Ultrasonography for disorders of sex development in pediatrics.
PubDate: 2025
Source: Front Pediatr
PMID: 40196161

🔹 Article 4:
Title: Worldwide Innovative Network (WIN) Consortium in Personalized Cancer Medicine: Bringing next-generation precision oncology to patients.
PubDate: 2025 Mar 12
Source: Oncotarget
PMID: 40073368

🔹 Article 5:
Title: Biology and Management of Male-Bodied Athletes in Elite Female Sports.
PubDate: 2025 Feb 27
Source: Drug Test Anal
PMID: 40015957

🔹 Article 6:
Title: Computer simulation study of confined oblate hard ellip

## With the New Relations Datasets

In [ ]:
Entrez.email = "brian.c.tran@sjsu.edu"

def pubmed_check(entity1, entity2, max_results=3):
    """Check if there are PubMed articles for a pair of entities."""
    query = f"{entity1} AND {entity2}"
    try:
        handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
        record = Entrez.read(handle)
        handle.close()
        return len(record["IdList"])
    except HTTPError as e:
        print(f"HTTPError for query '{query}': {e}")
        return 0

hit_counts = []

for idx, row in tqdm(df_results.iterrows(), total=len(df_results), desc="🔍 Checking PubMed"):
    e1 = row["entity1"]
    e2 = row["entity2"]
    hit_count = pubmed_check(e1, e2)
    hit_counts.append(hit_count)
    time.sleep(0.4)

df_results["pubmed_hits"] = hit_counts


has_evidence = df_results[df_results["pubmed_hits"] > 0]
print(f"\n{len(has_evidence)} of {len(df_results)} pairs have PubMed evidence.")
has_evidence.to_csv("chemprot_with_pubmed_hits.csv", index=False)